# CS175: Semi-Supervised Neural Code Translation
**Team Members:** Elian Hijmans Malessy, Sterling Radisay, Sia Aggarwal

This notebook demonstrates our fine-tuned CodeT5-small model translating source code between Python and C++. We utilized a semi-supervised training pipeline incorporating Supervised Fine-Tuning (SFT), Backtranslation (BT), and Denoising Auto-Encoding (DAE).

Below, we load our best checkpoint directly from Hugging Face and evaluate it on a few sample code snippets to highlight both its strengths and limitations.

In [ ]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_REPO = "sradisay/codet5-small-semi-supervised-cs175"
tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
model = T5ForConditionalGeneration.from_pretrained(HF_REPO).to(device)

print("Model loaded successfully!")

In [ ]:
def translate_code(source_code, source_lang, target_lang):
    model.eval()

    prompt = f"Translate {source_lang} to {target_lang}: {source_code}"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=256,
            temperature=0.2,
            do_sample=True
        )

    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

### Example 1: Translation (Python -> C++)
Our model performs well on standard algorithmic functions, but struggles with whitespace and newlines. Regardless, code does compile


In [ ]:
python_snippet = """
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
"""

print("--- Original Python ---")
print(python_snippet.strip())

c_plus_plus_translation = translate_code(python_snippet, "Python", "C++")

print("\n--- Translated C++ ---")
print(c_plus_plus_translation)

### Example 2: Translation (C++ -> Python)
Here we translate a CPP function multiply to python

In [ ]:
cpp_snippet = """
int multiply(int a, int b) {\n    return a * b;\n}
"""

print("--- Original C++ ---")
print(cpp_snippet.strip())

python_translation = translate_code(cpp_snippet, "C++", "Python")

print("\n--- Translated Python ---")
print(python_translation)

### Conclusion
This brief demonstration showcases the model's ability to map cross-lingual syntax after semi-supervised training. For comprehensive details on our multi-phase training loops and full HumanEval-X functional testing, please refer to our final project report and the `src/` directory in this repository.